# Tool schemas

Claude cannot read your Python code. It learns about a tool from a **schema**: a description in
[JSON Schema](https://json-schema.org/), a general data-validation format the AI world adopted for
function arguments.

A tool spec has three parts:

| Key | What goes in |
|---|---|
| `name` | the tool name, usually the function name |
| `description` | what it does, when to use it, what it returns: 3-4 sentences |
| `input_schema` | JSON Schema of the arguments: type and description of each one, `required` list |

Writing schemas by hand is tedious, so the lesson suggests letting Claude write them:
paste the function and ask for "a valid JSON schema spec for the purposes of tool calling",
with Anthropic's tool use docs attached for the best practices.

Naming convention: `get_current_datetime` → `get_current_datetime_schema`.

## Setup

In [1]:
import inspect
import json
from datetime import datetime

from anthropic import Anthropic
from anthropic.types import ToolParam
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

## Let Claude write the schema

The function source goes into the prompt. A ` ```json ` prefill and a ` ``` ` stop sequence
return clean JSON (see `01_accessing_claude_with_the_api/06_structured_data`).

In [2]:
prompt = f"""
Write a valid JSON schema spec for the purposes of tool calling for this function.
Follow the best practices of Anthropic's tool use documentation: the description should be
3-4 sentences on what the tool does, when to use it and what it returns.

<function>
{inspect.getsource(get_current_datetime)}
</function>
"""

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": "```json"},
    ],
    stop_sequences=["```"],
)
text = "".join(block.text for block in response.content if block.type == "text")
print(json.dumps(json.loads(text), indent=2))

{
  "name": "get_current_datetime",
  "description": "Retrieves the current date and time formatted according to the specified format string. Use this tool when you need to know the present date and time, such as for timestamping events, scheduling tasks, or providing temporal context to users. The tool returns a string representation of the current datetime in the requested format, defaulting to 'YYYY-MM-DD HH:MM:SS' if no format is specified.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string that specifies how to format the datetime output (e.g., '%Y-%m-%d %H:%M:%S' for '2024-01-15 14:30:45'). Defaults to '%Y-%m-%d %H:%M:%S' if not provided.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
}


## The schema

The lesson's version, wrapped in `ToolParam`. The wrapper changes nothing at runtime; it lets the type
checker catch a wrong key before the API does. The lesson's description is one sentence, so it is
expanded here to the 3-4 sentences the lesson itself recommends.

In [3]:
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time formatted according to the specified format. "
        "Use it whenever the answer depends on the current date or time, for example to "
        "compute a date relative to today. "
        "Returns the formatted date and time as a string."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "A string specifying the format of the returned datetime. "
                    "Uses Python's strftime format codes, e.g. '%Y-%m-%d' or '%H:%M'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

## Check: send it to the API

Not part of this lesson, a preview of the next ones. The API validates the schema, and for a question
that needs the current time Claude answers with a **tool request** instead of text
(`stop_reason` is `tool_use`). Nothing runs the function yet.

In [4]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": "What time is it right now?"}],
    tools=[get_current_datetime_schema],
)
print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "text":
        print("text:", block.text)
    elif block.type == "tool_use":
        print("tool_use:", block.name, block.input)

stop_reason: tool_use
tool_use: get_current_datetime {}


Next lesson: reading these message blocks and running the tool.